In [1]:
!pip install -q transformers peft bitsandbytes accelerate faiss-cpu rouge-score tqdm pylcs

  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 120.0 MB/s eta 0:00:00


In [2]:
!pip install -q -U torchao pylcs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.2 MB/s eta 0:00:00


In [3]:
# =============================================================================
# BOOTSTRAP: restore full pipeline state from Drive caches (~5-10 min, CPU ok)
# Run this first after ANY runtime restart.
# =============================================================================
import os, re as _re, json, pickle, gc
import numpy as np, pandas as pd, faiss
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize
try:
    import pylcs; HAVE_PYLCS = True
except ImportError:
    os.system('pip install -q pylcs')
    try: import pylcs; HAVE_PYLCS = True
    except Exception: HAVE_PYLCS = False

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
DATA_DIR, OUTPUT_DIR = BASE/'data', BASE/'outputs'
CACHE = OUTPUT_DIR/'mbr_cache'

# ---- data ----
train_df = pd.read_csv(DATA_DIR/'Train.csv')
val_df   = pd.read_csv(DATA_DIR/'Val.csv')
test_df  = pd.read_csv(DATA_DIR/'Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'SampleSubmission.csv'); SUB_COLS = list(sample_sub.columns)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input','output']).reset_index(drop=True)
questions_raw = combined['input'].astype(str).tolist()
answers_raw   = combined['output'].astype(str).tolist()
subsets_raw   = combined['subset'].astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
SUBSET_TO_LANG = {'Aka_Gha':'Akan (Ghana)','Amh_Eth':'Amharic (Ethiopia)','Eng_Eth':'English (Ethiopia)',
 'Eng_Gha':'English (Ghana)','Eng_Ken':'English (Kenya)','Eng_Uga':'English (Uganda)',
 'Lug_Uga':'Luganda (Uganda)','Swa_Ken':'Swahili (Kenya)'}
scorer_both = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False)

# ---- embeddings (all cached) ----
corpus_emb = np.load(CACHE/'emb_corpus.npy'); val_emb = np.load(CACHE/'emb_val.npy'); test_emb = np.load(CACHE/'emb_test.npy')
gem_corpus = np.load(CACHE/'gem_emb_corpus.npy'); gem_val = np.load(CACHE/'gem_emb_val.npy'); gem_test = np.load(CACHE/'gem_emb_test.npy')
ans_emb = np.load(CACHE/'emb_corpus_answers.npy')
bge_corpus = np.load(CACHE/'bge_corpus.npy'); bge_val = np.load(CACHE/'bge_val.npy'); bge_test = np.load(CACHE/'bge_test.npy')

# ---- indices ----
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i,s in enumerate(subsets_raw) if s==sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask]); out[sub]=(ix,mask)
    return out
lang_indices = build_lang_idx(corpus_emb); gem_lang_idx = build_lang_idx(gem_corpus)
qa_idx = build_lang_idx(ans_emb); bge_idx = build_lang_idx(bge_corpus)
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ---- pickled state ----
val_cands_all = pickle.load(open(CACHE/'val_cands.pkl','rb'))
val_prep, val_refscores = pickle.load(open(CACHE/'val_prep.pkl','rb'))
v4c, v4p, v4r = pickle.load(open(CACHE/'val_union4.pkl','rb'))
P = pickle.load(open(CACHE/'uni_rebuild.pkl','rb'))
llm_ans = json.load(open(OUTPUT_DIR/'gemini_mbr_llm_prog.json'))

# ---- core functions (canonical definitions) ----
K_CANDIDATES, K_LEG, CAP = 15, 20, 400
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())
def _lcs_py(a,b):
    if not a or not b: return 0
    dp=[0]*(len(b)+1)
    for ai in a:
        prev=0
        for j,bj in enumerate(b):
            cur=dp[j+1]; dp[j+1]=prev+1 if ai==bj else max(dp[j+1],dp[j]); prev=cur
    return dp[-1]
def lcs_tok(a,b):
    if HAVE_PYLCS:
        v={}
        for t in a:
            if t not in v: v[t]=len(v)
        for t in b:
            if t not in v: v[t]=len(v)
        return pylcs.lcs_sequence_length(''.join(chr(0x100+v[t]) for t in a), ''.join(chr(0x100+v[t]) for t in b))
    return _lcs_py(a,b)
def uni_r1(rt,ht):
    if not rt or not ht: return 0.0
    return 2*sum((Counter(rt)&Counter(ht)).values())/(len(rt)+len(ht))
def uni_rl(rt,ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt,ht)/(len(rt)+len(ht))
def get_same_lang_candidates(q_text,q_emb,subset,k=K_CANDIDATES,exclude_exact=True):
    qs=q_text.strip()
    if subset in lang_indices:
        idx,mask=lang_indices[subset]
        D,I=idx.search(np.asarray(q_emb,np.float32).reshape(1,-1),min(k+5,len(mask)))
        out=[]
        for d,li in zip(D[0],I[0]):
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            out.append({'answer':answers_raw[ci],'sim':float(d),'idx':ci})
            if len(out)>=k: break
        if out: return out
    return []
def union4(q_text,afri_q,gem_q,bge_q,subset,exclude_exact=True):
    qs=q_text.strip(); rrf={}
    for idx_map,emb in [(lang_indices,afri_q),(gem_lang_idx,gem_q),(qa_idx,afri_q),(bge_idx,bge_q)]:
        if subset not in idx_map: continue
        idx,mask=idx_map[subset]
        D,I=idx.search(np.asarray(emb,np.float32).reshape(1,-1),min(K_LEG+5,len(mask)))
        r=0
        for li in I[0]:
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            rrf[ci]=rrf.get(ci,0.0)+1.0/(60+r); r+=1
            if r>=K_LEG: break
    ranked=sorted(rrf.items(),key=lambda kv:-kv[1])[:35]
    return [{'answer':answers_raw[ci],'sim':float(np.dot(afri_q,corpus_emb[ci])),'idx':ci} for ci,_ in ranked]
def uni_prep(cands,max_tok=80):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    seen,dd,ddw={},[],[]
    for a,wi in zip(answers,w):
        k=a.strip().lower()
        if k in seen: ddw[seen[k]]+=wi
        else: seen[k]=len(dd); dd.append(a); ddw.append(wi)
    ddw=np.array(ddw); ddw/=ddw.sum(); n=len(dd)
    toks=[uni_toks(a)[:max_tok] for a in dd]
    if n==1: return dd,ddw,np.zeros(1),np.zeros(1)
    S1,SL=np.zeros((n,n)),np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            S1[i,j]=S1[j,i]=uni_r1(toks[i],toks[j]); SL[i,j]=SL[j,i]=uni_rl(toks[i],toks[j])
    return dd,ddw,S1@ddw,SL@ddw
def mbr_idx(ub,ddw,alpha,margin):
    if len(ddw)==1: return 0
    u=ub+alpha*ddw; b=int(np.argmax(u))
    return b if (b!=0 and u[b]-u[0]>margin) else 0
uni_prior={sub: float(np.median([len(uni_toks(answers_raw[i])) for i,s in enumerate(subsets_raw) if s==sub])) for sub in SUBSET_TO_LANG}
def uni_stitch(cands,lam,sub):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    p=Counter()
    for a,wi in zip(answers,w):
        for t,c in Counter(uni_toks(a)[:CAP]).items(): p[t]+=wi*c
    ref_len=max(lam*uni_prior[sub],1.0); seen,pool=set(),[]
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+',a):
            s=s.strip(); st=uni_toks(s)
            if len(st)<4: continue
            k=' '.join(st)
            if k not in seen: seen.add(k); pool.append((s,Counter(st),len(st)))
    if not pool: return answers[0]
    def ef1(cov,hl):
        mt=sum(min(c,p[t]) for t,c in cov.items())
        if hl==0 or mt==0: return 0.0
        pr_,rc=mt/hl,mt/ref_len; return 2*pr_*rc/(pr_+rc)
    chosen,cov,hl,cur=[],Counter(),0,0.0
    while pool:
        bi,bg=-1,0.0
        for i,(s,sc_,sl) in enumerate(pool):
            nc=cov.copy(); nc.update(sc_)
            g=ef1(nc,hl+sl)-cur
            if g>bg: bg,bi=g,i
        if bi<0: break
        s,sc_,sl=pool.pop(bi); chosen.append(s); cov.update(sc_); hl+=sl; cur+=bg
    return ' '.join(chosen) if chosen else answers[0]

# ---- tuned decisions from the unicode rebuild (hardcoded = reproducible) ----
choice = {'Aka_Gha':('2leg',0.10,0.010),'Amh_Eth':('2leg',0.05,99.0),'Eng_Eth':('2leg',0.05,99.0),
          'Eng_Gha':('4leg',0.20,0.050),'Eng_Ken':('2leg',0.05,99.0),'Eng_Uga':('2leg',0.05,99.0),
          'Lug_Uga':('2leg',0.05,99.0),'Swa_Ken':('2leg',0.05,99.0)}
uni_stitch_gate = {'Aka_Gha':{'use':True,'lam':0.70,'pool':'2leg'},
                   'Eng_Gha':{'use':True,'lam':0.85,'pool':'4leg'},
                   'Amh_Eth':{'use':True,'lam':0.70,'pool':'2leg'}}
print("STATE RESTORED:", len(combined), "corpus |", len(llm_ans), "LLM answers | all caches loaded")

Mounted at /content/drive
STATE RESTORED: 36501 corpus | 2618 LLM answers | all caches loaded


In [7]:
# =============================================================================
# EPOCH 2-3 RESUME + STRONG-LANGUAGE PROBE  (the scaling-hypothesis test)
# Epoch-1 adapter untouched at qwen-ft-health. New adapter: qwen-ft-health-e3.
# =============================================================================
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import time, gc, json, inspect, torch
from pathlib import Path
from datetime import datetime
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
!pip install -q trl
from trl import SFTTrainer, SFTConfig
def log(m): print(f"[{datetime.now().strftime('%H:%M:%S')}] {m}", flush=True)

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/multilingual-health-qa/outputs')
E1_DIR = OUTPUT_DIR / 'qwen-ft-health'
E3_DIR = OUTPUT_DIR / 'qwen-ft-health-e3'; E3_DIR.mkdir(parents=True, exist_ok=True)
torch.backends.cuda.matmul.allow_tf32 = True
log(f"GPU: {torch.cuda.get_device_name(0)} | free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

train_texts = json.load(open(OUTPUT_DIR / 'ft_train_data.json'))
dataset = Dataset.from_dict({"text": train_texts})

MID = 'Qwen/Qwen2.5-7B-Instruct'
tok = AutoTokenizer.from_pretrained(MID)
tok.pad_token = tok.eos_token; tok.padding_side = 'right'
base = AutoModelForCausalLM.from_pretrained(MID, dtype=torch.bfloat16, device_map='cuda:0')
base.config.use_cache = False
model = PeftModel.from_pretrained(base, str(E1_DIR), is_trainable=True)   # RESUME epoch-1 weights
model.gradient_checkpointing_enable(); model.enable_input_require_grads()
log("Resumed epoch-1 adapter, trainable")

kw = dict(output_dir=str(E3_DIR/'ckpt'), num_train_epochs=2,        # epochs 2 and 3
    per_device_train_batch_size=8, gradient_accumulation_steps=4,
    learning_rate=5e-5,                                             # halved for resume
    warmup_steps=30, lr_scheduler_type='cosine', max_grad_norm=0.3,
    bf16=True, gradient_checkpointing=True, optim='adamw_torch_fused',
    max_length=1024, max_seq_length=1024, dataset_text_field='text', packing=False,
    completion_only_loss=True, logging_steps=50, save_strategy='epoch',
    save_total_limit=2, report_to='none', seed=42)
valid = set(inspect.signature(SFTConfig.__init__).parameters)
dropped = [k for k in kw if k not in valid]
args = SFTConfig(**{k: v for k, v in kw.items() if k in valid})
tk = dict(model=model, args=args, train_dataset=dataset)
if 'completion_only_loss' in dropped:
    from trl import DataCollatorForCompletionOnlyLM
    tk['data_collator'] = DataCollatorForCompletionOnlyLM(
        response_template='<|im_start|>assistant\n', tokenizer=tok, mlm=False)
tv = set(inspect.signature(SFTTrainer.__init__).parameters)
tk['processing_class' if 'processing_class' in tv else 'tokenizer'] = tok

trainer = SFTTrainer(**tk)
log("Resuming: epochs 2-3, lr 5e-5, eff. batch 32")
t0 = time.time(); trainer.train()
log(f"Done in {(time.time()-t0)/60:.0f} min")
trainer.model.save_pretrained(str(E3_DIR)); tok.save_pretrained(str(E3_DIR))
del trainer; gc.collect(); torch.cuda.empty_cache()
log("Epoch-3 adapter saved")

# =============================================================================
# PROBE: epoch-3 generation on Lug_Uga + Eng_Uga holdout samples (300 each)
# =============================================================================
# needs bootstrap state — run bootstrap BEFORE this cell if fresh session
ft = PeftModel.from_pretrained(base, str(E3_DIR)); ft.eval()
@torch.no_grad()
def ft_generate(q, lang, cands, max_new=350):
    ctx = "\n".join(f"{k+1}. {c['answer']}" for k, c in enumerate(cands[:5]))
    msgs = [{"role":"system","content":
             f"You are a multilingual health expert. Answer health questions based on the "
             f"reference information provided. Use the EXACT words and phrases from the "
             f"references when possible. Be complete and accurate. Answer in {lang}."},
            {"role":"user","content": f"Question: {q}\n\nReference answers:\n{ctx}"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to('cuda:0')
    out = ft.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

import numpy as np
from tqdm import tqdm
PROBE_N = 300
results = {}
for sub, lang in [('Lug_Uga','Luganda (Uganda)'), ('Eng_Uga','English (Uganda)')]:
    idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
            and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()][1::2][:PROBE_N]
    pp = OUTPUT_DIR / f'ftqwen_e3_{sub}_probe.json'
    pa = json.load(open(pp)) if pp.exists() else {}
    for n, i in enumerate(tqdm(idxs, desc=f"E3 {sub}")):
        if str(i) in pa: continue
        pa[str(i)] = ft_generate(val_qs[i], lang, val_cands_all[i])
        if (n+1) % 25 == 0: json.dump(pa, open(pp,'w'))
    json.dump(pa, open(pp,'w'))
    tag, a, m = choice[sub]
    g_w, c_w = [], []
    for i in idxs:
        rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
        gt = uni_toks(pa[str(i)])[:CAP]
        dd, ddw, u1, uL = uni_prep(val_cands_all[i])
        ct = uni_toks(dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)])[:CAP]
        g_w.append(0.37*uni_r1(rt,gt)+0.37*uni_rl(rt,gt))
        c_w.append(0.37*uni_r1(rt,ct)+0.37*uni_rl(rt,ct))
    results[sub] = (np.mean(g_w), np.mean(c_w))
    print(f"{sub}: E3-gen={np.mean(g_w):.4f}  comb_mbr={np.mean(c_w):.4f}  Δ={np.mean(g_w)-np.mean(c_w):+.4f}")
print("\nEpoch-1 reference: Lug_Uga gen=0.5231 vs 0.6124 (Δ=-0.0892)")
print("VERDICT: scaling road OPEN if E3 Lug_Uga Δ improves by >0.03 vs epoch-1's -0.089;")
print("         CLOSED if Δ barely moves -> Brainiac's edge is retrieval-side.")

[17:18:06] GPU: NVIDIA A100-SXM4-80GB | free: 43.6 GB


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[17:18:14] Resumed epoch-1 adapter, trainable


Adding EOS to train dataset:   0%|          | 0/29814 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/29814 [00:00<?, ? examples/s]

...

[22:40:48] Done in 322 min
[22:40:49] Epoch-3 adapter saved


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
E3 Lug_Uga: 100%|██████████| 300/300 [1:28:47<00:00, 17.76s/it]


Lug_Uga: E3-gen=0.5243  comb_mbr=0.6111  Δ=-0.0868


E3 Eng_Uga: 100%|██████████| 300/300 [52:48<00:00, 10.56s/it]


Eng_Uga: E3-gen=0.6285  comb_mbr=0.6370  Δ=-0.0086

Epoch-1 reference: Lug_Uga gen=0.5231 vs 0.6124 (Δ=-0.0892)
VERDICT: scaling road OPEN if E3 Lug_Uga Δ improves by >0.03 vs epoch-1's -0.089;
         CLOSED if Δ barely moves -> Brainiac's edge is retrieval-side.
